# **Universal Notebook Environment Setup**

In [ ]:
import os
import sys
import wandb

# --- AUTOMATIC ENVIRONMENT SETUP ---
KAGGLE_RUN = os.path.exists('/kaggle/working')

if KAGGLE_RUN:
    print("Running on Kaggle. Setting up paths...")
    # Kaggle Secret for W&B
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

    # Create a symlink so /content/artifacts works on Kaggle
    os.makedirs('/content', exist_ok=True)
    if not os.path.exists('/content/artifacts'):
        # Map Kaggle input artifacts to Colab path
        os.system('ln -s /kaggle/input /content/artifacts')
else:
    print("Running on Google Colab.")
    # Colab Secret for W&B
    try:
        from google.colab import userdata
        wandb_api_key = userdata.get('WANDB_API_KEY')
        wandb.login(key=wandb_api_key)
    except Exception as e:
        print("W&B Secret not found, you may need to login manually.")
        wandb.login()

# **Fetch Augmented Images and Model From W&B**

In [ ]:
import os
import sys
import wandb

# Initialize a single W&B run for environment setup
run = wandb.init(project="pcb-defect-detection", job_type="setup")

print("--- Downloading Dataset Artifact ---")
# 1. Download Dataset
artifact_dataset = run.use_artifact('pcb-augmented-dataset:latest', type='dataset')
dataset_dir = artifact_dataset.download()
print(f"Dataset ready at: {os.path.abspath(dataset_dir)}")

print("\n--- Downloading Core Model Artifact ---")
# 2. Download Core Model Code
artifact_code = run.use_artifact('pcb-core-models:latest', type='model')
model_dir = artifact_code.download()

# Add to sys.path to allow immediate import
if model_dir not in sys.path:
    sys.path.insert(0, model_dir)
print(f"Core model ready at: {model_dir}")

run.finish()

# **Device Setup & W&B Tracking Initialization**

In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
import wandb
from torch.utils.data import Dataset, DataLoader

# Initialize the experiment tracking run
run = wandb.init(
    project="pcb-defect-detection",
    name="Model_Architecture_Training",
    notes="Training the MobileViT + LeYOLO architecture and metric logging with validation",
    config={
        "epochs" : 150,
        "batch_size" : 16,
        "image_size" : 480,
        "loss" : "v8DetectionLoss",
        "optimizer" : "AdamW|lr=1e-3|weight_decay=1e-4",
        "scheduler" : "CosineAnnealingLR|T_max=150|eta_min=1e-6",
        "conf_threshold" : 0.05,
        "iou_threshold" : 0.45,
        "train/val_split" : "80/20",
        "multiplier": 2.0,
        "neck_depth": 4,
        "use_sppf": True,
        "grad_accum_steps": 2,
        "amp": True,
    },
)
config = wandb.config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# **Data Loader From W&B**

In [ ]:
import os
import cv2
import torch
from torch.utils.data import Dataset, DataLoader

class PCBDataset(Dataset):
    def __init__(self, img_dir, label_dir, img_size=config.image_size):
        self.img_dir   = img_dir
        self.label_dir = label_dir
        self.img_size  = img_size
        self.img_names = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        # Load and preprocess image
        img_path = os.path.join(self.img_dir, self.img_names[idx])
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))

        # Normalize and convert to tensor (CHW format)
        img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0

        # Load YOLO format labels
        label_path = os.path.join(self.label_dir, self.img_names[idx].replace('.jpg', '.txt'))
        boxes, labels = [], []
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        labels.append(int(float(parts[0])))  # handles '5.0' format
                        boxes.append([float(x) for x in parts[1:]])

        targets = {
            "boxes":   torch.tensor(boxes,  dtype=torch.float32),
            "labels":  torch.tensor(labels, dtype=torch.int64),
            "raw_img": img  # kept for W&B visualization
        }
        return img_tensor, targets


def collate_fn(batch, device):
    images   = torch.stack([item[0] for item in batch])
    raw_imgs = [item[1]["raw_img"] for item in batch]

    batch_idx_list, cls_list, box_list = [], [], []

    for b_idx, item in enumerate(batch):
        targets   = item[1]
        num_boxes = len(targets["labels"])
        if num_boxes > 0:
            batch_idx_list.append(torch.full((num_boxes,), b_idx, dtype=torch.long))
            cls_list.append(targets["labels"].unsqueeze(1))
            box_list.append(targets["boxes"])

    # Fixed: plain torch.cat — no broken markdown hyperlinks
    if len(batch_idx_list) > 0:
        batch_dict = {
            'batch_idx': torch.cat(batch_idx_list, dim=0).to(device),
            'cls':       torch.cat(cls_list,       dim=0).to(device),
            'bboxes':    torch.cat(box_list,        dim=0).to(device)
        }
    else:
        batch_dict = {
            'batch_idx': torch.empty(0,       dtype=torch.long).to(device),
            'cls':       torch.empty((0, 1),  dtype=torch.long).to(device),
            'bboxes':    torch.empty((0, 4),  dtype=torch.float32).to(device)
        }

    return images, batch_dict, raw_imgs


# ── DATASET & DATALOADER SETUP ─────────────────────────────────────────────────
DATA_DIR     = os.path.join(dataset_dir, 'train')
full_dataset = PCBDataset(
    os.path.join(DATA_DIR, "images"),
    os.path.join(DATA_DIR, "labels")
)

# 80/20 train/val split — fixed seed for reproducibility
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size   = total_size - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Collate_fn now receives device via lambda — no global dependency
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, device)
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,  # order doesn't matter for evaluation
    collate_fn=lambda b: collate_fn(b, device)
)

print(f"Dataset split — Train: {train_size} | Val: {val_size}")

# **LeYOLO Tune**

In [ ]:
import importlib.util
import os
import torch
import torch.nn as nn

def load_module(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Unable to load module {module_name} from {file_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

model_mobilevit_path = os.path.join(model_dir, "mobilevit_xxs.py")
model_leyolo_path = os.path.join(model_dir, "leyolo_head.py")

mobilevit_xxs = load_module("mobilevit_xxs", model_mobilevit_path)
leyolo_head = load_module("leyolo_head", model_leyolo_path)

class SPPF(nn.Module):
    def __init__(self, c1, c2, k=5):
        super().__init__()
        c_ = c1 // 2
        self.cv1 = nn.Conv2d(c1, c_, 1, 1, 0, bias=False)
        self.bn1 = nn.BatchNorm2d(c_)
        self.act1 = nn.SiLU()
        self.cv2 = nn.Conv2d(c_ * 4, c2, 1, 1, 0, bias=False)
        self.bn2 = nn.BatchNorm2d(c2)
        self.act2 = nn.SiLU()
        self.m = nn.MaxPool2d(kernel_size=k, stride=1, padding=k // 2)

    def forward(self, x):
        x = self.act1(self.bn1(self.cv1(x)))
        y1 = self.m(x)
        y2 = self.m(y1)
        return self.act2(self.bn2(self.cv2(torch.cat((x, y1, y2, self.m(y2)), 1))))

class ECA(nn.Module):
    def __init__(self, channels: int, k_size: int = 3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)
        y = self.conv(y.squeeze(-1).transpose(-1, -2))
        y = self.sigmoid(y).transpose(-1, -2).unsqueeze(-1)
        return x * y

class TunedLeNeckBlock(nn.Module):
    def __init__(self, c1, c2, k=3, e=None, stride=1, pw=True, use_attention: bool = True):
        super().__init__()
        c_mid = e if e is not None else c1
        self.residual = c1 == c2 and stride == 1

        layers = []
        if pw and c_mid != c1:
            layers.extend([
                nn.Conv2d(c1, c_mid, kernel_size=1, bias=False),
                nn.BatchNorm2d(c_mid),
                nn.SiLU(),
            ])

        layers.extend([
            nn.Conv2d(c_mid, c_mid, kernel_size=k, stride=stride, padding=k // 2, groups=c_mid, bias=False),
            nn.BatchNorm2d(c_mid),
            nn.SiLU(),
            nn.Conv2d(c_mid, c2, kernel_size=1, bias=False),
            nn.BatchNorm2d(c2),
        ])

        self.layers = nn.Sequential(*layers)
        self.attention = ECA(c2) if use_attention else nn.Identity()

    def forward(self, x):
        out = self.layers(x)
        if self.residual:
            out = out + x
        return self.attention(out)

class TunedLeNeck(nn.Module):
    def __init__(self, width, multiplier=2.0, depth=4):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2)

        self.p4_up = [TunedLeNeckBlock(c1=width[1] + width[2], c2=int(64 * multiplier), e=int(128 * multiplier), k=5)]
        self.p4_up += [TunedLeNeckBlock(c1=int(64 * multiplier), c2=int(64 * multiplier), e=int(128 * multiplier), k=5)
                       for _ in range(int(2 * depth))]
        self.p4_up = nn.Sequential(*self.p4_up)

        self.p3_up = [TunedLeNeckBlock(c1=int(64 * multiplier) + width[0], c2=int(32 * multiplier), e=int(64 * multiplier) + width[0], k=3, pw=False)]
        self.p3_up += [TunedLeNeckBlock(c1=int(32 * multiplier), c2=int(32 * multiplier), e=int(96 * multiplier), k=3)
                       for _ in range(int(2 * depth))]
        self.p3_up = nn.Sequential(*self.p3_up)

        self.p2_up = TunedLeNeckBlock(c1=int(32 * multiplier), c2=int(16 * multiplier), e=int(32 * multiplier), k=3, pw=False)

        self.p3_downsampling = nn.Conv2d(int(32 * multiplier), int(64 * multiplier), kernel_size=3, stride=2, padding=1)

        self.p4_down = [TunedLeNeckBlock(c1=int(64 * multiplier) + int(64 * multiplier), c2=int(64 * multiplier), e=int(128 * multiplier), k=5)]
        self.p4_down += [TunedLeNeckBlock(c1=int(64 * multiplier), c2=int(64 * multiplier), e=int(128 * multiplier), k=5)
                         for _ in range(int(2 * depth))]
        self.p4_down = nn.Sequential(*self.p4_down)
        self.p4_downsampling = nn.Conv2d(int(64 * multiplier), int(96 * multiplier), kernel_size=3, stride=2, padding=1)

        self.p5_down = [TunedLeNeckBlock(c1=int(96 * multiplier) + width[2], c2=int(96 * multiplier), e=int(96 * multiplier) + width[2], k=5)]
        self.p5_down += [TunedLeNeckBlock(c1=int(96 * multiplier), c2=int(96 * multiplier), e=int(192 * multiplier), k=5)
                         for _ in range(int(2 * depth))]
        self.p5_down = nn.Sequential(*self.p5_down)

    def forward(self, x):
        p3, p4, p5 = x
        p5_up = torch.cat(tensors=[self.up(p5), p4], dim=1)
        p4_up = self.p4_up(p5_up)

        p3_up = torch.cat(tensors=[self.up(p4_up), p3], dim=1)
        p3_up = self.p3_up(p3_up)
        p2_up = self.p2_up(self.up(p3_up))
        p4_down = self.p3_downsampling(p3_up)

        p4_down = torch.cat(tensors=[p4_down, p4_up], dim=1)
        p4_down = self.p4_down(p4_down)

        p5_down = self.p4_downsampling(p4_down)
        p5_down = torch.cat(tensors=[p5_down, p5], dim=1)
        p5_down = self.p5_down(p5_down)

        return p2_up, p3_up, p4_down, p5_down

class TunedLeYOLO(nn.Module):
    def __init__(self, core_module, num_classes, multiplier=2.0, depth=4, image_size: int = 480, use_sppf: bool = True):
        super().__init__()
        self.net = core_module.MobileViTXXSBackbone(image_size=image_size)
        width = [48, 64, 320]
        self.sppf = SPPF(width[2], width[2]) if use_sppf else nn.Identity()
        self.fpn = TunedLeNeck(width, multiplier=multiplier, depth=depth)

        img_dummy = torch.zeros(1, 3, image_size, image_size)
        self.head = core_module.LeHead(num_classes, (int(16 * multiplier), int(32 * multiplier), int(64 * multiplier), int(96 * multiplier)))
        self.head.stride = torch.tensor([image_size / x.shape[-2] for x in self.forward(img_dummy)])
        self.stride = self.head.stride
        self.head.initialize_biases()

    def forward(self, x):
        x = self.net(x)
        x[2] = self.sppf(x[2])
        x = self.fpn(x)
        return self.head(list(x))

# Configure tuned parameters from W&B config (fallback to defaults)
neck_multiplier = getattr(config, "multiplier", 2.0)
neck_depth = getattr(config, "neck_depth", 4)
use_sppf = getattr(config, "use_sppf", True)

model = TunedLeYOLO(
    leyolo_head,
    num_classes=6,
    multiplier=neck_multiplier,
    depth=neck_depth,
    image_size=config.image_size,
    use_sppf=use_sppf,
).to(device)
print("Tuned LeYOLO initialized from core architectures.")

# **Custom Model Evaluation Function**

In [ ]:
%pip install -q ultralytics

In [8]:
import torch
import torchvision.ops as ops
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from ultralytics.utils.tal import make_anchors
from ultralytics.utils.loss import v8DetectionLoss

def _build_decoder(model, device):
    class DummyModelConfig:
        def __init__(self, full_model, target_device):
            self._full_model = full_model
            self.device = target_device
            class Args:
                box, cls, dfl, cls_pw = 7.5, 0.5, 1.5, 1.0
            self.args = Args()
            class MockDetectHead:
                def __init__(self, head):
                    self.stride, self.nc, self.no, self.device = head.stride, head.nc, head.no, target_device
                    self.reg_max = head.ch
                    self.use_dfl = True
            self.model = [MockDetectHead(full_model.head)]

        def parameters(self):
            return self._full_model.parameters()
    loss_fn = v8DetectionLoss(DummyModelConfig(model, device))

    def decode(preds):
        if isinstance(preds, list):
            box_ch = model.head.ch * 4
            pred_boxes = torch.cat(
                [p[:, :box_ch, :, :].reshape(p.shape[0], box_ch, -1) for p in preds],
                dim=2,
            )
            pred_scores = torch.cat(
                [p[:, box_ch:, :, :].reshape(p.shape[0], model.head.nc, -1) for p in preds],
                dim=2,
            )
            feats = preds
            pred_distri = pred_boxes.permute(0, 2, 1).contiguous()
            pred_scores = pred_scores.permute(0, 2, 1).contiguous().sigmoid()
            anchor_points, stride_tensor = make_anchors(feats, model.head.stride, 0.5)
            pred_bboxes = loss_fn.bbox_decode(anchor_points, pred_distri) * stride_tensor
            return pred_bboxes, pred_scores

        if isinstance(preds, dict):
            pred_boxes = preds["boxes"]
            pred_scores = preds["scores"]
            feats = preds["feats"]
            pred_distri = pred_boxes.permute(0, 2, 1).contiguous()
            pred_scores = pred_scores.permute(0, 2, 1).contiguous().sigmoid()
            anchor_points, stride_tensor = make_anchors(feats, model.head.stride, 0.5)
            pred_bboxes = loss_fn.bbox_decode(anchor_points, pred_distri) * stride_tensor
            return pred_bboxes, pred_scores

        if torch.is_tensor(preds):
            if preds.dim() != 3 or preds.shape[1] != 4 + model.head.nc:
                raise ValueError(f"Unexpected prediction tensor shape: {preds.shape}")
            pred_boxes_xywh = preds[:, :4, :].permute(0, 2, 1).contiguous()
            pred_scores = preds[:, 4:, :].permute(0, 2, 1).contiguous()
            xy, wh = pred_boxes_xywh[..., :2], pred_boxes_xywh[..., 2:]
            pred_bboxes = torch.cat([xy - wh / 2, xy + wh / 2], dim=-1)
            return pred_bboxes, pred_scores

        raise TypeError(f"Unsupported prediction type: {type(preds)}")

    return decode

def _ensure_val_loader(val_source):
    if hasattr(val_source, "__iter__") and hasattr(val_source, "batch_size"):
        return val_source
    return torch.utils.data.DataLoader(
        val_source,
        batch_size=getattr(config, "batch_size", 8),
        shuffle=False,
        collate_fn=lambda b: collate_fn(b, device),
    )

def evaluate_model(
    model,
    val_source,
    device,
    conf_threshold: float | None = None,
    iou_threshold: float | None = None,
 ) -> dict:
    conf_threshold = conf_threshold if conf_threshold is not None else getattr(config, "conf_threshold", 0.25)
    iou_threshold = iou_threshold if iou_threshold is not None else getattr(config, "iou_threshold", 0.45)
    model.eval()
    decoder = _build_decoder(model, device)
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox', class_metrics=True)

    val_loader = _ensure_val_loader(val_source)
    with torch.no_grad():
        for batch in val_loader:
            if len(batch) == 3:
                images, target_dict, _ = batch
            else:
                images, target_dict = batch
            images = images.to(device)
            _, _, H, W = images.shape
            decoded_preds = model(images)
            pred_bboxes, pred_scores = decoder(decoded_preds)
            preds_list, targets_list = [], []

            for b in range(images.shape[0]):
                gt_mask = target_dict['batch_idx'] == b
                gt_boxes_norm = target_dict['bboxes'][gt_mask]
                gt_labels = target_dict['cls'][gt_mask].squeeze(-1).to(torch.int64)

                if len(gt_boxes_norm) > 0:
                    x_c, y_c, bw, bh = gt_boxes_norm.unbind(1)
                    gt_boxes_xyxy = torch.stack([
                        (x_c - bw / 2) * W, (y_c - bh / 2) * H,
                        (x_c + bw / 2) * W, (y_c + bh / 2) * H,
                    ], dim=1).to(device)
                else:
                    gt_boxes_xyxy = torch.empty((0, 4), device=device)

                targets_list.append({
                    "boxes":  gt_boxes_xyxy,
                    "labels": gt_labels.to(device),
                })

                pred_boxes_xyxy = pred_bboxes[b]
                pred_scores_b = pred_scores[b]
                max_scores, class_indices = pred_scores_b.max(dim=1)
                conf_mask = max_scores > conf_threshold
                f_boxes  = pred_boxes_xyxy[conf_mask]
                f_scores = max_scores[conf_mask]
                f_labels = class_indices[conf_mask]

                if len(f_boxes) > 0:
                    keep = ops.nms(f_boxes, f_scores, iou_threshold=iou_threshold)
                    max_det = 100
                    keep = keep[:max_det]
                    preds_list.append({
                        "boxes":  f_boxes[keep],
                        "scores": f_scores[keep],
                        "labels": f_labels[keep].to(torch.int64),
                    })
                else:
                    preds_list.append({
                        "boxes":  torch.empty((0, 4),                   device=device),
                        "scores": torch.empty((0,),                     device=device),
                        "labels": torch.empty((0,), dtype=torch.int64,  device=device),
                    })

            metric.update(preds_list, targets_list)

    results = metric.compute()
    model.train()
    return results

# **Create Visualize Predictions for Validation**

In [ ]:
from matplotlib import patches, pyplot as plt


def visualize_predictions(
    model,
    dataset,
    device,
    num_images=4,
    conf_threshold: float | None = None,
    iou_threshold: float | None = None,
 ):
    conf_threshold = conf_threshold if conf_threshold is not None else getattr(config, "conf_threshold", 0.25)
    iou_threshold = iou_threshold if iou_threshold is not None else getattr(config, "iou_threshold", 0.45)
    model.eval()
    decoder = _build_decoder(model, device)
    import math
    import random
    num_images = min(num_images, len(dataset))
    indices = random.sample(range(len(dataset)), num_images)
    cols = 2
    rows = max(1, math.ceil(num_images / cols))
    fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(15, 5 * rows))
    axes = axes.flatten()
    with torch.no_grad():
        for i, idx in enumerate(indices):
            image, target = dataset[idx]
            image = image.to(device)
            _, H, W = image.shape
            preds = model(image.unsqueeze(0))
            pred_bboxes, pred_scores = decoder(preds)
            pred_boxes_xyxy = pred_bboxes[0]
            pred_scores_b = pred_scores[0]
            max_scores, class_indices = pred_scores_b.max(dim=1)
            conf_mask = max_scores > conf_threshold
            f_boxes  = pred_boxes_xyxy[conf_mask]
            f_scores = max_scores[conf_mask]
            f_labels = class_indices[conf_mask]
            if len(f_boxes) > 0:
                keep = ops.nms(f_boxes, f_scores, iou_threshold=iou_threshold)
                f_boxes  = f_boxes[keep]
                f_scores = f_scores[keep]
                f_labels = f_labels[keep]

            # Convert image to numpy for visualization
            img = image.permute(1, 2, 0).cpu().numpy()
            img = (img * 255).astype(np.uint8)
            axes[i].imshow(img)
            axes[i].axis('off')
            axes[i].set_title(f"Image {idx}")

            # Plot GT boxes (green)
            gt_boxes_norm = target.get('bboxes', target.get('boxes', torch.empty((0, 4))))
            for box in gt_boxes_norm:
                x_c, y_c, bw, bh = box
                x1 = (x_c - bw / 2) * W
                y1 = (y_c - bh / 2) * H
                x2 = (x_c + bw / 2) * W
                y2 = (y_c + bh / 2) * H
                rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                         linewidth=2, edgecolor='g', facecolor='none')
                axes[i].add_patch(rect)

            # Plot predicted boxes (red)
            for box in f_boxes:
                x1, y1, x2, y2 = box.cpu().numpy()
                rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                         linewidth=2, edgecolor='r', facecolor='none')
                axes[i].add_patch(rect)
    for j in range(len(indices), len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.close(fig)
    return fig

In [ ]:
# Quick analysis: label stats + single-batch sanity check
import numpy as np

def quick_analysis(dataset, model, device, num_samples: int = 200):
    sample_count = min(num_samples, len(dataset))
    label_values = []
    boxes_norm = []
    for i in range(sample_count):
        _, target = dataset[i]
        lbls = target.get("labels", torch.tensor([], dtype=torch.int64))
        label_values.append(lbls.cpu().numpy())
        boxes_norm.append(target.get("boxes", torch.empty((0, 4))).cpu().numpy())
    all_labels = np.concatenate(label_values) if label_values else np.array([])
    all_boxes = np.concatenate(boxes_norm) if boxes_norm else np.zeros((0, 4))

    print("Label stats:")
    if all_labels.size:
        unique, counts = np.unique(all_labels, return_counts=True)
        print("  unique labels:", dict(zip(unique.tolist(), counts.tolist())))
    else:
        print("  no labels found")
    if all_boxes.size:
        print("  box min/max:", all_boxes.min(axis=0), all_boxes.max(axis=0))
    else:
        print("  no boxes found")

    model.eval()
    image, target = dataset[0]
    image = image.to(device)
    preds = model(image.unsqueeze(0))
    pred_bboxes, pred_scores = _build_decoder(model, device)(preds)
    print("Prediction stats:")
    print("  pred_bboxes range:", pred_bboxes.min().item(), pred_bboxes.max().item())
    print("  pred_scores range:", pred_scores.min().item(), pred_scores.max().item())
    model.train()

dataset_for_analysis = (
    val_dataset if "val_dataset" in globals()
    else train_dataset if "train_dataset" in globals()
    else full_dataset
    if "full_dataset" in globals() else None
 )
if dataset_for_analysis is None:
    print("Dataset not found. Run the data loader cell first.")
else:
    quick_analysis(dataset_for_analysis, model, device)

Dataset not found. Run the data loader cell first.


# **Overfit Loop & W&B Training Tracking**

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.ops as ops
from torch.optim.lr_scheduler import CosineAnnealingLR
from ultralytics.utils.loss import v8DetectionLoss

# Optional: reduce fragmentation on CUDA
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

def unwrap_model(model):
    return model.module if hasattr(model, "module") else model

# 1. DUMMY WRAPPER (Required for Hybrid Model + Native Loss)
class DummyModelConfig:
    def __init__(self, full_model, target_device):
        self._full_model = full_model
        self.device = target_device

        class Args:
            box, cls, dfl, cls_pw = 7.5, 0.5, 1.5, 1.0
        self.args = Args()

        class MockDetectHead:
            def __init__(self, head):
                self.stride = head.stride
                self.nc = head.nc
                self.no = head.no
                self.device = target_device
                self.reg_max = head.ch
                self.use_dfl = True
        # Ultralytics loss expects model.model[-1] to be the Detect head
        self.model = [MockDetectHead(full_model.head)]

    def parameters(self):
        return self._full_model.parameters()

# 2. LOSS, OPTIMIZER & SCHEDULER
dummy_config = DummyModelConfig(model, device)
yolo_loss_fn = v8DetectionLoss(dummy_config)

epochs = config.epochs
optimizer  = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler  = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

scaler = torch.amp.GradScaler(enabled=bool(config.amp))
grad_accum_steps = max(1, int(config.grad_accum_steps))

# 3. BEST MODEL TRACKING
best_map50   = 0.0
best_epoch   = 0

print("Launching Training Run (150 epochs)...")

# 4. TRAINING LOOP
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0

    for batch_idx, (images, target_dict, raw_imgs) in enumerate(train_loader):
        images = images.to(device)
        optimizer.zero_grad(set_to_none=True) if batch_idx % grad_accum_steps == 0 else None

        with torch.amp.autocast(enabled=bool(config.amp), device_type='cuda'):
            predictions = model(images)
            if isinstance(predictions, list):
                base_model = unwrap_model(model)
                box_ch = base_model.head.ch * 4
                pred_boxes = torch.cat(
                    [p[:, :box_ch, :, :].reshape(p.shape[0], box_ch, -1) for p in predictions],
                    dim=2,
                )
                pred_scores = torch.cat(
                    [p[:, box_ch:, :, :].reshape(p.shape[0], base_model.head.nc, -1) for p in predictions],
                    dim=2,
                )
                predictions = {
                    "boxes": pred_boxes,
                    "scores": pred_scores,
                    "feats": predictions,
                }
            loss, loss_items = yolo_loss_fn(predictions, target_dict)
            loss = loss.sum() / grad_accum_steps

        scaler.scale(loss).backward()

        if (batch_idx + 1) % grad_accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        epoch_loss += loss.item() * grad_accum_steps

    scheduler.step()
    avg_epoch_loss = epoch_loss / len(train_loader)
    current_lr     = scheduler.get_last_lr()[0]

    wandb.log({
        "Train_Loss":    avg_epoch_loss,
        "Learning_Rate": current_lr,
        "Epoch":         epoch
    })

    if epoch % 10 == 0 or epoch == epochs - 1:
        print(f"\nRunning Evaluation — Epoch [{epoch}/{epochs}]...")
        val_metrics = evaluate_model(
            model,
            val_loader,
            device,
            conf_threshold=config.conf_threshold,
            iou_threshold=config.iou_threshold,
        )
        viz_img     = visualize_predictions(
            model,
            val_dataset,
            device,
            epoch,
            conf_threshold=config.conf_threshold,
            iou_threshold=config.iou_threshold,
        )

        current_map50 = val_metrics['map_50'].item()

        wandb.log({
            "Val_mAP_50":             current_map50,
            "Val_Recall":             val_metrics['mar_100'].item(),
            "Val_mAP_50_95":          val_metrics['map'].item(),
            "Validation/Predictions": wandb.Image(viz_img, caption=f"Epoch {epoch}"),
            "Epoch":                  epoch
        })

        print(f"Epoch [{epoch}/{epochs}] | Loss: {avg_epoch_loss:.4f} "
              f"| mAP@0.5: {current_map50:.4f} "
              f"| Recall: {val_metrics['mar_100'].item():.4f} "
              f"| LR: {current_lr:.6f}")

        if current_map50 > best_map50:
            best_map50 = current_map50
            best_epoch = epoch
            torch.save(model.state_dict(), "mobilevit_leyolo_best.pt")
            print(f"  New best model saved — mAP@0.5: {best_map50:.4f} at epoch {best_epoch}")

torch.save(model.state_dict(), "mobilevit_leyolo_final.pt")
wandb.save("mobilevit_leyolo_best.pt")
wandb.save("mobilevit_leyolo_final.pt")
wandb.finish()

print(f"\nTraining Complete!")